# NUPACK RNA Tile Design (Multi-tile, simplified)

This notebook designs RNA sequences for **all tiles** in an input file where each tile already contains:

- a **sequence line** (with spaces to indicate segments)
- a **dot-paren secondary structure line** (matching the full sequence length, ignoring spaces)

It will:
1. Parse and **count tiles**
2. For each tile:
   - clean sequence/structure (remove spaces, T→U)
   - validate `len(seq) == len(structure)`
   - run **NUPACK tube_design (3 trials)**
   - pick the best trial by **lowest ensemble defect**
   - **re-insert the original spaces** into INPUT/STRUCTURE/OUTPUT for readability
3. Write a single output file named **`Output_sequence`** containing all tiles.

## Expected input format

```
(optional header line)
*******TILE0*******
<sequence line with spaces>
<structure line with spaces>
*******TILE1*******
...
```


In [1]:
from nupack import *
from pathlib import Path
from IPython.display import display, HTML
import re


In [2]:
def clean_sequence(seq_raw: str) -> str:
    # Strip whitespace, force RNA alphabet (T->U).
    return ''.join(seq_raw.split()).upper().replace('T', 'U')

def clean_structure(struct_raw: str) -> str:
    # Keep only valid dot-paren characters; drop spaces/linebreaks/etc.
    allowed = set('().')
    return ''.join(ch for ch in struct_raw if ch in allowed)

def apply_spacing(unspaced: str, seg_lengths):
    '''
    Re-insert spaces according to segment lengths.
    Example: seg_lengths=[14,1,9,...] -> "AAAA... A BBBBBBBBB ..."
    '''
    parts = []
    i = 0
    for L in seg_lengths:
        parts.append(unspaced[i:i+L])
        i += L
    if i != len(unspaced):
        raise ValueError(f"Segmentation lengths sum to {i}, but string length is {len(unspaced)}")
    return ' '.join(parts)

def report_lengths(seq: str, struct: str, label: str = ""):
    if label:
        print(f"[{label}]")
    print(f"Sequence length : {len(seq)}")
    print(f"Structure length: {len(struct)}")
    open_count  = struct.count('(')
    close_count = struct.count(')')
    if open_count != close_count:
        print(f"WARNING: structure parentheses UNBALANCED: '('={open_count}, ')'={close_count}")
    if len(seq) != len(struct):
        raise ValueError(f"Length mismatch: len(seq)={len(seq)} vs len(struct)={len(struct)}")

def parse_tiles_file(path: str):
    '''
    Expected format:
      [optional first non-empty line: header string]
      *******TILE0*******
      <sequence line>
      <structure line>
      (blank lines)
      *******TILE1*******
      ...

    Returns:
      header_line (str|None), tiles (list[dict{name, seq_raw, struct_raw}])
    '''
    p = Path(path)
    lines = p.read_text().splitlines()
    lines = [ln.rstrip() for ln in lines]

    first_idx = next((i for i, ln in enumerate(lines) if ln.strip()), None)
    if first_idx is None:
        raise ValueError("Input file is empty.")

    header_line = None
    i = first_idx

    if not lines[i].strip().startswith("*******TILE"):
        header_line = lines[i].strip()
        i += 1

    tiles = []
    tile_header_re = re.compile(r"^\*{7}TILE(\d+)\*{7}\s*$")

    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue

        m = tile_header_re.match(lines[i].strip())
        if not m:
            i += 1
            continue

        tile_name = f"TILE{m.group(1)}"
        i += 1

        while i < len(lines) and not lines[i].strip():
            i += 1
        if i >= len(lines):
            raise ValueError(f"{tile_name}: missing sequence line.")
        seq_raw = lines[i]
        i += 1

        while i < len(lines) and not lines[i].strip():
            i += 1
        if i >= len(lines):
            raise ValueError(f"{tile_name}: missing structure line.")
        struct_raw = lines[i]
        i += 1

        tiles.append({"name": tile_name, "seq_raw": seq_raw, "struct_raw": struct_raw})

    if not tiles:
        raise ValueError("No tiles found. Check file formatting (TILE headers).")

    return header_line, tiles


In [3]:
# -----------------------------
# Config
# -----------------------------
# User requested local-folder style:
INPUT_FILE = "Generated_Tiles_9x12_108.txt"   # change to your filename if needed
TRIALS = 3

# Model: RNA, 37 C, 1.0 M Na+, NUPACK3-like ensemble
model = Model(material='rna',
              celsius=37,
              sodium=1.0,
              ensemble='some-nupack3')

# Global "prevent" patterns as SOFT constraints (penalize, do not forbid)
prevent_list = ['A4','C4','G4','U4','K6','M6','R6','S6','W6','Y6']
# Lower weight => easier feasibility; increase weight if you want stronger avoidance
soft_constraints = [Pattern(prevent_list, weight=1.0)]

# Design options
options = DesignOptions(f_stop=0.02, seed=42)


In [4]:
# -----------------------------
# Parse input and count tiles
# -----------------------------
header_line, tiles = parse_tiles_file(INPUT_FILE)

print("Header line:", header_line)
print("Tiles found:", len(tiles))
print("First 10 tiles:", [t["name"] for t in tiles[:10]])


Header line: /Users/wyssuser/HMS Dropbox/Liangxiao Chen/ASU/dsRNA_brick/code/python/SelectedPool_from_graph_out_complement.txt
Tiles found: 108
First 10 tiles: ['TILE0', 'TILE1', 'TILE2', 'TILE3', 'TILE4', 'TILE5', 'TILE6', 'TILE7', 'TILE8', 'TILE9']


In [5]:
# -----------------------------
# Run design for ALL tiles
# -----------------------------
OUT_FILE = Path("Output_sequence")

# We'll stream-write results so partial progress is saved even if something fails mid-run
with OUT_FILE.open("w") as f:
    if header_line:
        f.write(header_line + "\n\n")

    for t_i, tile in enumerate(tiles, start=1):
        tile_name = tile["name"]
        print(f"\n==== [{t_i}/{len(tiles)}] {tile_name} ====")

        # Preserve segmentation from the input sequence line
        seq_tokens = [tok for tok in tile["seq_raw"].split() if tok]
        seg_lengths = [len(tok) for tok in seq_tokens]

        # Clean for NUPACK
        seq = clean_sequence(tile["seq_raw"])
        structure = clean_structure(tile["struct_raw"])

        # Validate
        report_lengths(seq, structure, label=tile_name)

        # Build targets
        a = Domain(seq, name=f'{tile_name}_a')
        RNA_seq = TargetStrand([a], name=f'{tile_name}_RNA_seq')
        target = TargetComplex([RNA_seq], structure, name=f'{tile_name}_target')

        tube = TargetTube(on_targets={target: 1e-6},
                          off_targets=SetSpec(max_size=1),
                          name=f'{tile_name}_tube')

        design = tube_design(tubes=[tube],
                             soft_constraints=soft_constraints,
                             model=model,
                             options=options)

        # Run trials (catch exceptions per-tile, continue to next)
        try:
            results = design.run(trials=TRIALS)
        except Exception as e:
            print(f"ERROR running NUPACK for {tile_name}: {e}")
            f.write(f"*******{tile_name}*******\n\n")
            f.write(f"ERROR\t{e}\n\n")
            continue

        # Pick best by ensemble defect
        completed = [(i, res) for i, res in enumerate(results) if res is not None]
        if not completed:
            msg = "No completed trials."
            print("ERROR:", msg)
            f.write(f"*******{tile_name}*******\n\n")
            f.write(f"ERROR\t{msg}\n\n")
            continue

        trial_defects = []
        for idx, res in completed:
            ed = float(res.defects.ensemble_defect)
            trial_defects.append((idx, ed))
            print(f"Trial {idx+1:>2}: ensemble defect = {ed:.6g}")

        best_idx, best_defect = min(trial_defects, key=lambda x: x[1])
        best_result = results[best_idx]
        print(f"Best trial = {best_idx+1} (ensemble defect {best_defect:.6g})")

        # Extract designed sequence (NUPACK4 API)
        try:
            designed_seq_unspaced = str(best_result.to_analysis(RNA_seq))
        except TypeError:
            # in case to_analysis behaves like a mapping
            designed_seq_unspaced = str(best_result.to_analysis[RNA_seq])

        # Re-insert spaces for readability
        input_seq_spaced = apply_spacing(seq, seg_lengths)
        input_struct_spaced = apply_spacing(structure, seg_lengths)
        output_seq_spaced = apply_spacing(designed_seq_unspaced, seg_lengths)

        # Write block
        f.write(f"*******{tile_name}*******\n\n")
        f.write("INPUT_SEQUENCE\t" + input_seq_spaced + "\n\n")
        f.write("INPUT_STRUCTURE\t\n" + input_struct_spaced + "\n\n")
        f.write("OUTPUT_SEQUENCE\t" + output_seq_spaced + "\n\n")

print("Wrote:", OUT_FILE.resolve())



==== [1/108] TILE0 ====
[TILE0]
Sequence length : 116
Structure length: 116
Trial  1: ensemble defect = 0.0285754
Trial  2: ensemble defect = 0.0272439
Trial  3: ensemble defect = 0.0264062
Best trial = 3 (ensemble defect 0.0264062)

==== [2/108] TILE1 ====
[TILE1]
Sequence length : 224
Structure length: 224
Trial  1: ensemble defect = 0.0340787
Trial  2: ensemble defect = 0.0342941
Trial  3: ensemble defect = 0.0400191
Best trial = 1 (ensemble defect 0.0340787)

==== [3/108] TILE2 ====
[TILE2]
Sequence length : 224
Structure length: 224
Trial  1: ensemble defect = 0.0325869
Trial  2: ensemble defect = 0.0318419
Trial  3: ensemble defect = 0.0315904
Best trial = 3 (ensemble defect 0.0315904)

==== [4/108] TILE3 ====
[TILE3]
Sequence length : 224
Structure length: 224
Trial  1: ensemble defect = 0.0335927
Trial  2: ensemble defect = 0.0331547
Trial  3: ensemble defect = 0.0338781
Best trial = 2 (ensemble defect 0.0331547)

==== [5/108] TILE4 ====
[TILE4]
Sequence length : 224
Structure